In [1]:
from transformers import EncoderDecoderModel

model = EncoderDecoderModel.from_encoder_decoder_pretrained(
    "bert-base-uncased",  # encoder
    "bert-base-uncased"   # decoder
)

/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14075.56it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ig

In [5]:
import torch
import torch.nn as nn
from transformers import BertModel, BertConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using: {device}")

# 1. Encoder
class BertEncoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        for param in self.bert.parameters():
            param.requires_grad = False
        self.project_down = nn.Linear(768, latent_dim)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        z = self.project_down(cls)
        return z

# 2. Parallel Decoder
class ParallelDecoder(nn.Module):
    def __init__(self, latent_dim=256, seq_len=128, vocab_size=30522):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_dim = 768
        self.project_up = nn.Linear(latent_dim, seq_len * 768)
        config = BertConfig.from_pretrained("bert-base-uncased")
        config.is_decoder = False
        self.bert = BertModel.from_pretrained("bert-base-uncased", config=config)
        self.to_logits = nn.Linear(768, vocab_size)

    def forward(self, z):
        x = self.project_up(z)
        x = x.view(-1, self.seq_len, self.hidden_dim)
        out = self.bert(inputs_embeds=x)
        logits = self.to_logits(out.last_hidden_state)
        return logits

# 3. Test
encoder = BertEncoder(latent_dim=256).to(device)
decoder = ParallelDecoder(latent_dim=256).to(device)

dummy_ids = torch.randint(0, 30522, (2, 128)).to(device)
dummy_mask = torch.ones(2, 128, dtype=torch.long).to(device)

with torch.no_grad():
    z = encoder(dummy_ids, dummy_mask)
    logits = decoder(z)

print(f"z shape:      {z.shape}")
print(f"logits shape: {logits.shape}")
print("forward pass ok")

using: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13957.17it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26924.73it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  

z shape:      torch.Size([2, 256])
logits shape: torch.Size([2, 128, 30522])
forward pass ok


In [6]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# encode a real sentence
text = "the cat sat on the mat"
inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=128,
    padding="max_length",
    truncation=True
)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

# forward pass
with torch.no_grad():
    z = encoder(input_ids, attention_mask)
    logits = decoder(z)
    pred_ids = logits.argmax(-1)  # [1, 128]

# decode both
original = tokenizer.decode(input_ids[0], skip_special_tokens=True)
predicted = tokenizer.decode(pred_ids[0], skip_special_tokens=True)

print(f"original:  {original}")
print(f"predicted: {predicted}")

original:  the cat sat on the mat
predicted: forestry mortal'' mortal forestry' mortal mortal'''''' mortal''' anthropology mortal' mortal'' mortal' mortal mortal' mortal mortal mortal' mortal mortal mortal mortal mortal mortal mortal mortal'' mortal''' mortal'''' mortal'' mortal' mortal [unused70]' mortal' mortal''' mortal mortal'' mortal mortal mortal'' mortal mortal mortal mortal mortal' andrews'' mortal' mortal' mortal mortal mortal mortal mortal' mortal'' mortal mortal' mortal' mortal'''' [unused70] andrews'' andrews mortal mortal mortal andrews mortal mortal mortal mortal mortal andrews' mortal'''


In [7]:
import torch.optim as optim
from datasets import load_dataset

# load small slice first
ds = load_dataset("wikitext", "wikitext-103-raw-v1")
small_train = ds["train"].select(range(5000))  # 5000 samples to start
small_val   = ds["validation"]

print(f"train: {len(small_train)} samples")
print(f"val:   {len(small_val)} samples")

train: 5000 samples
val:   3760 samples


In [8]:
from torch.utils.data import DataLoader

# tokenize
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length",
        return_tensors="pt"
    )

# filter empty strings first
small_train = small_train.filter(lambda x: len(x["text"].strip()) > 10)
small_val   = small_val.filter(lambda x: len(x["text"].strip()) > 10)

# tokenize
train_tok = small_train.map(tokenize, batched=True)
val_tok   = small_val.map(tokenize, batched=True)

# set format for pytorch
train_tok.set_format(type="torch", columns=["input_ids", "attention_mask"])
val_tok.set_format(type="torch", columns=["input_ids", "attention_mask"])

# dataloaders
train_loader = DataLoader(train_tok, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_tok,   batch_size=16, shuffle=False)

print(f"train batches: {len(train_loader)}")
print(f"val batches:   {len(val_loader)}")

Map: 100%|██████████| 2454/2454 [00:00<00:00, 14414.28 examples/s]

train batches: 199
val batches:   154


In [ ]:
import torch.nn.functional as F
from torch.optim import AdamW

# optimizer — only train projection layers and decoder
# encoder BERT is frozen so skip it
optimizer = AdamW(
    list(encoder.project_down.parameters()) +
    list(decoder.parameters()),
    lr=1e-4
)

EPOCHS = 3
VOCAB_SIZE = 30522
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    
    # ── training ──────────────────────────────────────
    encoder.train()
    decoder.train()
    train_loss = 0
    
    for step, batch in enumerate(train_loader):
        input_ids     = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        # forward
        z      = encoder(input_ids, attention_mask)
        logits = decoder(z)
        
        # reconstruction loss
        loss = F.cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            input_ids.view(-1),
            ignore_index=0  # ignore padding
        )
        
        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        if step % 50 == 0:
            print(f"epoch {epoch+1} step {step}/{len(train_loader)} | loss {loss.item():.4f}")
    
    avg_train = train_loss / len(train_loader)
    
    # ── validation ────────────────────────────────────
    encoder.eval()
    decoder.eval()
    val_loss = 0
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            z      = encoder(input_ids, attention_mask)
            logits = decoder(z)
            
            loss = F.cross_entropy(
                logits.view(-1, VOCAB_SIZE),
                input_ids.view(-1),
                ignore_index=0
            )
            val_loss += loss.item()
    
    avg_val = val_loss / len(val_loader)
    
    print(f"\nepoch {epoch+1} done | train loss {avg_train:.4f} | val loss {avg_val:.4f}\n")
    
    # save best
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            "encoder": encoder.state_dict(),
            "decoder": decoder.state_dict(),
        }, "stage1_best.pt")
        print(f"saved best model at val loss {best_val_loss:.4f}")

epoch 1 step 0/199 | loss 10.3006
